In [7]:
import pandas as pd

hh = pd.read_csv(
    "../data/raw/household/household_power_consumption.txt",
    sep=";", na_values=["?"], low_memory=False,
)
hh["timestamp"] = pd.to_datetime(hh["Date"] + " " + hh["Time"], format="%d/%m/%Y %H:%M:%S")
hh = hh.drop(columns=["Date", "Time"]).set_index("timestamp").sort_index()

ap = pd.read_csv("../data/raw/appliances/energydata_complete.csv", parse_dates=["date"])
ap = ap.set_index("date").sort_index()

In [8]:
na = hh["Global_active_power"].isna()
# longueur des séquences de NaN consécutifs
groups = (na != na.shift()).cumsum()
gap_len = na.groupby(groups).sum()
gap_len = gap_len[gap_len > 0]
print(gap_len.describe())
print(gap_len.sort_values(ascending=False).head(10))   # les plus longs trous, en minutes

count      71.000000
mean      365.901408
std      1251.468043
min         1.000000
25%         1.000000
50%         1.000000
75%         3.000000
max      7226.000000
Name: Global_active_power, dtype: float64
Global_active_power
138    7226
140    5237
14     3723
100    3305
118    3129
126    2027
104     891
32       83
78       70
34       47
Name: Global_active_power, dtype: int64


In [9]:
def clean_appliances(df, freq="1h"):
    df = df.drop(columns=["rv1", "rv2"], errors="ignore")
    df.columns = [c.lower() for c in df.columns]
    df = df[~df.index.duplicated(keep="first")]

    sum_cols = ["appliances", "lights"]
    mean_cols = [c for c in df.columns if c not in sum_cols]
    agg = {**{c: "sum" for c in sum_cols}, **{c: "mean" for c in mean_cols}}
    return df.resample(freq).agg(agg)

In [10]:
def flag_outliers(s, k=3.0):
    q1, q3 = s.quantile([0.25, 0.75])
    iqr = q3 - q1
    return (s < q1 - k * iqr) | (s > q3 + k * iqr)

In [11]:
from sklearn.model_selection import TimeSeriesSplit
def cv_splits(n_splits=5):
    return TimeSeriesSplit(n_splits=n_splits)

In [14]:
def save_splits(train, val, test, name, out="data/processed"):
    train.to_parquet(f"{out}/{name}_train.parquet")
    val.to_parquet(f"{out}/{name}_val.parquet")
    test.to_parquet(f"{out}/{name}_test.parquet")

In [ ]:
from sklearn.base import BaseEstimator

def train(train_df, val_df, params) -> BaseEstimator:  # retourne un objet avec .predict(X)
    # Votre code d'entraînement
    return model